# Papaia quick start: a Google ADK agent with a local LLM

This notebook runs a [Google ADK](https://github.com/google/adk-python) agent with one tool, `factor`, on a CPU-only Colab runtime, **without a GPU or an API key**.
The tool is declared with `@required_fn(pip_install=["sympy"])`, so [Papaia](https://github.com/Max-ChenFei/papaia) places it on the local node and runs it in its own virtual environment; ADK sees only the generated stub.
The LLM is `qwen3:0.6b`, served locally by [Ollama](https://ollama.com).

Run the cells in order (*Runtime → Run all*). The first run takes a few minutes to install packages and download the model (about 0.5 GB).

## 1. Install Papaia and Google ADK

In [ ]:
# Papaia creates each tool's environment with Python's venv module, which needs ensurepip
!python -c "import ensurepip" 2>/dev/null || apt-get -qq install -y "python$(python -c 'import sys; print(f"{sys.version_info[0]}.{sys.version_info[1]}")')-venv" > /dev/null
!pip install -q "git+https://github.com/Max-ChenFei/papaia.git" "google-adk==2.14.1" "litellm==1.109.1"

## 2. Start Ollama and download the model

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen3:0.6b 2> /dev/null && ollama list

## 3. Declare the compute resources

One local node, with no GPU and no remote machine.

In [ ]:
%%writefile exec_nodes.toml
# One local node: no GPU, no remote machine.
[nodes.local]
location = "local"
cpus = 2
memory = 4

## 4. Write the agent

`factor` is an ordinary Python function with a Papaia decorator. The agent registers it with ADK as usual.

In [ ]:
%%writefile agent.py
"""Quick start: a Google ADK agent with one Papaia tool, a local LLM, no GPU, no API key."""

import asyncio
import os

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.genai import types

from papaia import required_fn


@required_fn(pip_install=["sympy"])
def factor(expression: str) -> str:
    """Factor a polynomial expression.

    Args:
        expression: A polynomial in Python syntax, e.g. "x**2 - 1".
    """
    import sys

    import sympy

    return f"{sympy.factor(expression)} (computed in the environment {sys.prefix})"


agent = Agent(
    name="math_assistant",
    model=LiteLlm(model="ollama_chat/" + os.environ.get("MODEL", "qwen3:0.6b")),
    instruction="You are a math assistant. Always call the factor tool to factor a polynomial; never factor it yourself. /no_think",
    tools=[factor],
    generate_content_config=types.GenerateContentConfig(temperature=0),
)


async def main() -> None:
    runner = InMemoryRunner(agent=agent, app_name="quickstart")
    session = await runner.session_service.create_session(app_name="quickstart", user_id="user")
    question = types.Content(role="user", parts=[types.Part(text="Factor x**2 - 1.")])
    async for event in runner.run_async(user_id="user", session_id=session.id, new_message=question):
        for part in event.content.parts if event.content and event.content.parts else []:
            if part.function_call:
                print(f"[tool call]   {part.function_call.name}({part.function_call.args})")
            if part.function_response:
                print(f"[tool result] {part.function_response.response}")
            if part.text and not part.thought and event.is_final_response():
                print("\nANSWER:", part.text.strip())


asyncio.run(main())

## 5. Compile, place, deploy, and run

`papaia run` places `factor` on the local node, creates its virtual environment with `sympy`, starts its service, replaces the function with a stub, and runs the agent.
The tool result shows the environment in which `factor` ran.

In [ ]:
!papaia run agent.py -c exec_nodes.toml

## Next steps

To move a tool to another node, change only `exec_nodes.toml`; see the [Docker example](https://github.com/Max-ChenFei/papaia/tree/main/use_cases/weather_demo_docker), which simulates a local and a remote GPU node.
More examples and the API reference are in the [repository](https://github.com/Max-ChenFei/papaia).